Access to the dataset and Fetching descriptions

In [4]:
import pandas as pd
import requests
import re
import os
from dotenv import load_dotenv

load_dotenv() 

# === CONFIG ===
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
INPUT_FILES = {
    "top": "dataset/top_lib.csv",
    "middle": "dataset/middle_lib.csv",
    "bottom": "dataset/bottom_lib.csv"
}
OUTPUT_CSV = "Ori_lib_list.csv"

# === FUNCTIONS ===

def extract_owner_repo(url):
    """Extracts 'owner/repo' from a GitHub URL."""
    match = re.search(r"github\.com/([^/]+/[^/]+)", url)
    return match.group(1) if match else None

def fetch_description(owner_repo):
    """Fetches the GitHub repo description using the API."""
    url = f"https://api.github.com/repos/{owner_repo}"
    headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json().get("description", "")
    return f"Error: {response.status_code}"

# === LOAD AND COMBINE DATA ===

all_dfs = []

for source, file_path in INPUT_FILES.items():
    df = pd.read_csv(file_path)
    df['owner_repo'] = df['Repository_URL'].apply(extract_owner_repo)
    df['source'] = source
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df = combined_df.dropna(subset=["owner_repo"])  # drop rows with no valid repo

# === FETCH DESCRIPTIONS ===

unique_repos = combined_df['owner_repo'].unique()
descriptions = {repo: fetch_description(repo) for repo in unique_repos}

# === PREPARE OUTPUT ===

output_df = combined_df[['owner_repo', 'source']].drop_duplicates()
output_df['description'] = output_df['owner_repo'].map(descriptions)
output_df = output_df.rename(columns={"owner_repo": "library"})

# Clean library name (keep only repo name)
output_df['library'] = output_df['library'].astype(str).apply(lambda x: x.split('/')[-1])

# === SAVE OUTPUT ===

output_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved descriptions with source to {OUTPUT_CSV}")


✅ Saved descriptions with source to Ori_lib_list.csv


Check number of null description

In [5]:
import pandas as pd

df = pd.read_csv("Ori_lib_list.csv")

# Group by source, then count NaNs in 'description' column
nan_counts_by_source = df.groupby('source')['description'].apply(lambda x: x.isna().sum())

print("Number of NaN descriptions by source:")
print(nan_counts_by_source)




Number of NaN descriptions by source:
source
bottom    20
middle     9
top       12
Name: description, dtype: int64


In [9]:
import pandas as pd

lib_list = pd.read_csv("Ori_lib_list.csv")

summary = pd.DataFrame({
    "Total": lib_list["source"].value_counts(),
    "Valid": lib_list[
        lib_list["description"].notna() &
        (lib_list["description"] != "Error: 404")
    ]["source"].value_counts(),
    "Error_404": lib_list[
        lib_list["description"] == "Error: 404"
    ]["source"].value_counts(),
    "NaN": lib_list[
        lib_list["description"].isna()
    ]["source"].value_counts(),
}).fillna(0).astype(int)

print(summary)

        Total  Valid  Error_404  NaN
source                              
bottom    500    467         13   20
middle    500    488          3    9
top       500    486          2   12


In [13]:
import pandas as pd

# Load files
ori = pd.read_csv("Ori_lib_list.csv")
clean = pd.read_csv("lib_list.csv")

# Libraries that survived the previous cleaning
clean_libs = set(clean["library"])

# Libraries that were removed previously
removed = ori[~ori["library"].isin(clean_libs)]

print(f"Removed previously: {len(removed)}")

# Of those removed, which were 404?
old_404 = removed[removed["description"] == "Error: 404"]

print(f"\nOld 404 count: {len(old_404)}")
print(old_404[["library", "source"]].sort_values("source"))

Removed previously: 50

Old 404 count: 9
                                 library  source
1020  ampersand-collection-pouchdb-mixin  bottom
1053                            autocode  bottom
1068                       baucis-vivify  bottom
1148                             conflab  bottom
1151                         consolation  bottom
1378                             hookies  bottom
1410                              isansi  bottom
262                     file-entry-cache     top
264                           flat-cache     top


In [12]:
import pandas as pd

lib_list = pd.read_csv("lib_list.csv")

lib_list = lib_list[lib_list["description"] != "Error: 404"].reset_index(drop=True)

lib_list.to_csv("lib_list.csv", index=False)

print(f"Remaining libraries: {len(lib_list)}")

Remaining libraries: 1450


In [1]:
import pandas as pd

# Load files
lib_list = pd.read_csv("lib_list.csv")
rec_df = pd.read_csv("0_rec_RQ1_v2.csv")

# หา index ของ descriptions ที่เป็น Error: 404
bad_desc_idx = lib_list.index[
    lib_list["description"] == "Error: 404"
]

# ลบแถวใน recommendation file ที่อ้างถึง index เหล่านั้น
rec_df = rec_df[~rec_df["Desc_Index"].isin(bad_desc_idx)]

# Save
rec_df.to_csv("0_rec_RQ1_v2.csv", index=False)

print(f"Removed {len(bad_desc_idx)} bad descriptions")
print(f"Remaining rows: {len(rec_df)}")

Removed 0 bad descriptions
Remaining rows: 7250


In [5]:
import pandas as pd
df = pd.read_csv("0_rec_RQ1_v2.csv")

summary = df.groupby("Run")["Desc_Index"].count()
print(summary)

Run
1    1450
2    1450
3    1450
4    1450
5    1450
Name: Desc_Index, dtype: int64


In [7]:
import pandas as pd

df = pd.read_csv("0_rec_RQ1_v2.csv")

df = df.sort_values(by="Desc_Index").reset_index(drop=True)

df.to_csv("0_rec_RQ1_v2.csv", index=False)